# `sklm`: one fine-tune, every tabular task

`sklm` exposes a fine-tuned autoregressive language model as a family of scikit-learn estimators —
a classifier, a regressor, a missing-value imputer, and an imbalanced-learn oversampler. They are
not four separate models. They are four *views* of **one** fitted model, and this notebook explains
that shared core before the task notebooks (`01`–`04`) put it to work.

## The one mechanism

1. Each table row is **serialized to text** — for example the JSON
   `{"sepal length": 5.1, ..., "target": "setosa"}`.
2. A small autoregressive LM is **fine-tuned** on those strings, but the **column order of each row
   is permuted on every epoch**.

An autoregressive model factorizes the probability of a token sequence as

$$ \log p_\theta(t_1,\dots,t_T) = \sum_{i=1}^{T} \log p_\theta\!\left(t_i \mid t_1,\dots,t_{i-1}\right). $$

Serializing a row in a *fixed* column order would only teach the model that one autoregressive
factorization. By permuting the order $\pi$ across epochs, the model is trained on many
factorizations of the same row, so it learns to condition **any** column on **any subset** of the
others:

$$ p\!\left(x_j \mid x_S\right)\qquad\text{for arbitrary } j \text{ and } S \subseteq \{1,\dots,d\}\setminus\{j\}. $$

That single conditional is everything the estimators need. Each one is just a choice of *which*
columns go into the prompt $x_S$ and *which* column $x_j$ the model produces:

| Estimator | Conditions on (prompt) | Produces (target) | How it reads the answer |
|-----------|------------------------|-------------------|-------------------------|
| `LanguageModelClassifier` | all features | the class label | **scores** each candidate label, ranks them |
| `LanguageModelRegressor` | all features | the numeric target | **generates** the value `n` times, averages |
| `LanguageModelImputer` | a row's observed cells | that row's missing cells | **generates** each missing value |
| `LanguageModelOverSampler` | a minority class label | the features | **generates** synthetic rows |

Two reading strategies cover all four: **scoring** a fixed candidate set (deterministic — the
classifier, and discretized numeric columns) and **generating** a value as text (the regressor,
imputer, oversampler). The rest of this notebook shows the core they share — the serialized text,
the permutation, and one live conditional query.

In [1]:
from random import Random

from sklearn.datasets import load_iris

from sklm import (
    Field,
    JSONSerializer,
    JupyterCallback,
    MLXBackend,
    ModelConfig,
    TabularLanguageModel,
    TrainingConfig,
)

SEED = 42

## What the model is trained on

A `Serializer` turns each row into the string the model sees; the default is JSON. Here is one Iris
row as a list of `Field`s and its serialized form — the exact text the LM is fine-tuned on.

In [2]:
row = [
    Field(name="sepal length", value=5.1, numeric=True),
    Field(name="sepal width", value=3.5, numeric=True),
    Field(name="petal length", value=1.4, numeric=True),
    Field(name="petal width", value=0.2, numeric=True),
    Field(name="target", value="setosa", numeric=False),
]
print(JSONSerializer().serialize(row))

{"sepal length": 5.1, "sepal width": 3.5, "petal length": 1.4, "petal width": 0.2, "target": "setosa"}


## The permutation

During training that same row is emitted under many different column orders — this is the
permutation augmentation that forces the model to learn $p(x_j \mid x_S)$ rather than one fixed
factorization. A few of the orderings the model would see across epochs:

In [3]:
rng = Random(SEED)
for _ in range(3):
    order = row[:]
    rng.shuffle(order)
    print(JSONSerializer().serialize(order))

{"petal width": 0.2, "sepal width": 3.5, "petal length": 1.4, "target": "setosa", "sepal length": 5.1}
{"petal width": 0.2, "petal length": 1.4, "sepal length": 5.1, "target": "setosa", "sepal width": 3.5}
{"petal width": 0.2, "sepal width": 3.5, "petal length": 1.4, "sepal length": 5.1, "target": "setosa"}


## One fine-tune, queried directly

`TabularLanguageModel` is the object every estimator wraps. Fitting it with no fixed target treats
*every* column as a potential target — the permutation does the rest. We fit on the full Iris table
(four measurements plus the species label) and keep it short; distilgpt2 on Iris is only an API
demonstration, not a strong model.

In [4]:
iris = load_iris(as_frame=True)
frame = iris.data.round(1)
frame["species"] = iris.target_names[iris.target]

lm = TabularLanguageModel(
    backend=MLXBackend(),
    serializer=JSONSerializer(),
    model=ModelConfig(model="mlx-community/distilgpt2"),
    training=TrainingConfig(epochs=40, batch_size=16),
    callback=JupyterCallback(),
    random_state=SEED,
).fit(frame)

## A live conditional query

Condition on just the two petal measurements and ask for the species distribution.
`predict_proba(known, target, candidates)` scores each candidate label by likelihood and
normalizes — exactly what `LanguageModelClassifier` does internally. The same fitted model can also
*generate* a column instead of scoring one; `05-conditional-queries` drives both paths in depth.

In [5]:
known = {"petal length (cm)": 1.4, "petal width (cm)": 0.2}
proba = lm.predict_proba(known, "species", list(iris.target_names))
for c, p in zip(iris.target_names, proba, strict=True):
    print(f"p(species={c} | small petals) = {p:.3f}")

p(species=setosa | small petals) = 0.892
p(species=versicolor | small petals) = 0.085
p(species=virginica | small petals) = 0.023


## Where to go next

The same fitted conditional answers every task; each notebook just picks which columns to condition
on and how to read the answer.

- `01-iris-classifier` — score the label set (classification)
- `02-autompg-regressor` — predict a numeric target by scoring or sampling (regression)
- `03-tips-imputer` — fill a missing categorical cell on mixed-type data, beating KNN (imputation)
- `04-imbalanced-oversampler` — synthesize minority-class rows (oversampling)
- `05-conditional-queries` — drive `TabularLanguageModel` directly, both scoring and generation
- `06-optuna-search`, `07-stratified-cv` — tuning and evaluation with the scikit-learn contract
- `08-synthesizer` — sample whole rows from the learned joint $p(x_1,\dots,x_d)$